# Experimentation notebook

Interactive companion to the scripts. Run from the project root (`jupyter notebook notebooks/experimentation.ipynb`).

Sections:
1. Dataset sanity checks (counts, leakage check, sample crops)
2. Load the trained model and score the test split
3. Grad-CAM gallery + region attribution
4. Threshold sweep (precision / recall trade-off)


In [ ]:
import sys, os, json
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import config
print("project root:", ROOT)
print("architecture:", config.ARCHITECTURE, "| img size:", config.IMG_SIZE, "| frames/video:", config.FRAMES_PER_VIDEO)

## 1. Dataset sanity checks

In [ ]:
manifest = pd.read_csv(config.MANIFEST_PATH)
display(manifest.groupby(["split", "label"]).agg(crops=("crop", "count"), videos=("source_video", "nunique")))

# Leakage check: no source video may appear in more than one split.
leak = manifest.groupby("source_video")["split"].nunique()
print("videos in >1 split:", int((leak > 1).sum()), "(must be 0)")
print("landmark coverage:", f"{manifest['has_landmarks'].mean():.1%}")

In [ ]:
def show_crops(split="train", n=6):
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 4.8))
    for row, label in enumerate(config.CLASS_FOLDERS):
        files = sorted((config.DATASET_DIR / split / label).glob("*.jpg"))
        picks = np.random.default_rng(0).choice(len(files), size=min(n, len(files)), replace=False)
        for ax, i in zip(axes[row], picks):
            img = cv2.cvtColor(cv2.imread(str(files[i])), cv2.COLOR_BGR2RGB)
            ax.imshow(img); ax.set_title(label, fontsize=9); ax.axis("off")
    plt.tight_layout()

show_crops()

## 2. Load the model and score the test split

In [ ]:
import keras
from train import load_split

model = keras.models.load_model(config.MODEL_PATH)
class_names = json.load(open(config.CLASS_NAMES_PATH))
fake_idx = class_names.index("fake")

test_ds = load_split("test", config.BATCH_SIZE, shuffle=False, augment_jpeg=False)
probs = model.predict(test_ds, verbose=0)
y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds])
p_fake = probs[:, fake_idx]
is_fake = (y_true == fake_idx).astype(int)

from sklearn.metrics import roc_auc_score, accuracy_score
print(f"frame-level  AUC {roc_auc_score(is_fake, p_fake):.3f}   acc@0.5 {accuracy_score(is_fake, p_fake >= 0.5):.3f}   n={len(p_fake)}")

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.hist(p_fake[is_fake == 0], bins=30, alpha=0.7, label="real frames")
plt.hist(p_fake[is_fake == 1], bins=30, alpha=0.7, label="fake frames")
plt.axvline(config.THRESHOLD, color="k", ls="--", lw=1, label="threshold")
plt.xlabel("P(fake)"); plt.ylabel("frames"); plt.legend(); plt.title("Score distribution on the test split")
plt.tight_layout()

## 3. Grad-CAM gallery

Each row: face crop, Grad-CAM for the predicted class, top attributed regions.

In [ ]:
from explainability.gradcam import GradCAM
from explainability.explain import explain
from preprocessing.preprocessing import preprocess_face
from utils.image_utils import overlay_heatmap

cam = GradCAM(model)
files = list(test_ds.file_paths)
rng = np.random.default_rng(1)
picks = rng.choice(len(files), size=min(6, len(files)), replace=False)

fig, axes = plt.subplots(len(picks), 2, figsize=(6, 3 * len(picks)))
for (ax_a, ax_b), i in zip(axes, picks):
    img = cv2.imread(files[i])
    x = preprocess_face(img)
    pr = model.predict(x[None], verbose=0)[0]
    k = int(np.argmax(pr))
    heat, _ = cam.compute(x, k)
    exp = explain(heat, class_names[k], float(pr[k]), None, img.shape[0])
    ax_a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax_a.axis("off")
    ax_a.set_title(f"true={Path(files[i]).parent.name}", fontsize=9)
    ax_b.imshow(cv2.cvtColor(overlay_heatmap(img, heat), cv2.COLOR_BGR2RGB)); ax_b.axis("off")
    top = ", ".join(f"{r.label} {r.share:.0%}" for r in exp.regions[:2])
    ax_b.set_title(f"pred={class_names[k]} ({pr[k]:.2f}) · {top}", fontsize=8)
plt.tight_layout()

## 4. Threshold sweep

The default threshold is 0.5. If false alarms on real videos are costlier than missed fakes (or vice versa), move it.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for t in np.arange(0.1, 0.95, 0.05):
    pred = (p_fake >= t).astype(int)
    rows.append({"threshold": round(t, 2),
                 "precision_fake": precision_score(is_fake, pred, zero_division=0),
                 "recall_fake": recall_score(is_fake, pred, zero_division=0),
                 "f1_fake": f1_score(is_fake, pred, zero_division=0),
                 "accuracy": accuracy_score(is_fake, pred)})
sweep = pd.DataFrame(rows)
sweep.plot(x="threshold", y=["precision_fake", "recall_fake", "f1_fake", "accuracy"], figsize=(6, 3.2), grid=True)
plt.tight_layout()
sweep.loc[sweep.f1_fake.idxmax()]